In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS sales;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS categories;
DROP TABLE IF EXISTS reps;

CREATE TABLE categories (
    category_id   INTEGER PRIMARY KEY,
    category_name TEXT NOT NULL
);

CREATE TABLE products (
    product_id   INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category_id  INTEGER,
    price        REAL,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
);

CREATE TABLE reps (
    rep_id     INTEGER PRIMARY KEY,
    rep_name   TEXT NOT NULL,
    region     TEXT,
    manager_id INTEGER,
    hire_date  TEXT,
    FOREIGN KEY (manager_id) REFERENCES reps(rep_id)
);

CREATE TABLE sales (
    sale_id    INTEGER PRIMARY KEY,
    rep_id     INTEGER,
    product_id INTEGER,
    sale_date  TEXT,
    quantity   INTEGER,
    status     TEXT,
    FOREIGN KEY (rep_id)     REFERENCES reps(rep_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""")

categories = [
    (1, 'Electronics'),
    (2, 'Furniture'),
    (3, 'Stationery'),
    (4, 'Accessories'),   # no products, on purpose
]

products = [
    (101, 'Laptop',    1, 60000),
    (102, 'Monitor',   1, 15000),
    (103, 'Keyboard',  1,  2000),
    (104, 'Desk',      2,  8000),
    (105, 'Chair',     2,  8000),   # tie with Desk for most expensive
    (106, 'Notebook',  3,    50),
    (107, 'Pen',       3,    20),
    (108, 'Gift Card', None, 1000), # no category
]

reps = [
    (1, 'Arjun',  'South', None, '2018-01-10'),  # no manager
    (2, 'Bhavna', 'South', 1,    '2019-06-01'),
    (3, 'Charan', 'North', 1,    '2020-03-15'),
    (4, 'Divya',  'North', 3,    '2021-09-20'),
    (5, 'Eshan',  'West',  1,    '2022-11-05'),  # no sales
]

sales = [
    (1001, 2, 101, '2024-01-05',  1, 'Completed'),
    (1002, 2, 103, '2024-01-20',  5, 'Completed'),
    (1003, 3, 102, '2024-02-10',  3, 'Completed'),
    (1004, 3, 104, '2024-02-15',  2, 'Returned'),
    (1005, 4, 105, '2024-02-28',  4, 'Completed'),
    (1006, 2, 101, '2024-03-03',  1, 'Completed'),
    (1007, 1, 102, '2024-03-10',  2, 'Completed'),
    (1008, 4, 106, '2024-03-22', 10, 'Completed'),
    (1009, 3, 101, '2024-04-01',  1, 'Returned'),
    (1010, 2, 104, '2024-04-18',  2, 'Completed'),
    (1011, 4, 105, '2024-05-09',  3, 'Completed'),
    (1012, 1, 101, '2024-05-25',  1, 'Completed'),
    (1013, 3, 103, '2024-06-02',  8, 'Completed'),
    (1014, 4, 107, '2024-06-30', 20, 'Completed'),
    (1015, 2, 102, '2024-02-01',  1, 'Returned'),  # Bhavna, must be filtered out
]

cur.executemany("INSERT INTO categories VALUES (?,?)", categories)
cur.executemany("INSERT INTO products   VALUES (?,?,?,?)", products)
cur.executemany("INSERT INTO reps       VALUES (?,?,?,?,?)", reps)
cur.executemany("INSERT INTO sales      VALUES (?,?,?,?,?,?)", sales)
conn.commit()

def q(sql):
    """Run a query and RETURN the result as a DataFrame."""
    return pd.read_sql_query(sql, conn)

print("DB ready. Use q('SELECT ...') to run queries.")

DB ready. Use q('SELECT ...') to run queries.


In [4]:
# List every product name and its category name.
# Include products with no category (Gift Card). Sort by product name.
q("""
SELECT p.product_name,c.category_name from products p 
LEFT JOIN categories c on p.category_id = c.category_id
ORDER BY p.product_name ASC ;
""")

,product_name,category_name
0,Chair,Furniture
1,Desk,Furniture
2,Gift Card,None
3,Keyboard,Electronics
4,Laptop,Electronics
5,Monitor,Electronics
6,Notebook,Stationery
7,Pen,Stationery


In [10]:
# For each category show category name and number of products.
# Categories with no products (Accessories) must show 0.
# Sort by count descending, then category name.
q("""
SELECT c.category_name , COUNT(p.product_id) as 'cnt' from categories c
LEFT join products p on p.category_id = c.category_id
group by c.category_name
order by cnt DESC, c.category_name ASC ;
""")

,category_name,cnt
0,Electronics,3
1,Furniture,2
2,Stationery,2
3,Accessories,0


In [12]:
# Show each region and its total quantity sold, counting ONLY Completed sales,
# only for regions whose total quantity is above 10.
q("""
SELECT r.region , SUM(s.quantity) as 'total_qty' from sales s
join reps r on  r.rep_id = s.rep_id
where s.status='Completed'
group by r.region
having total_qty >10
""")

,region,total_qty
0,North,48
1,South,12


In [14]:
# Find products whose price is above the average product price.
# Return product name and price, highest first.
q("""
SELECT product_name, price from products where price > (select AVG(price) from products)
order by price DESC;
""")

,product_name,price
0,Laptop,60000.0
1,Monitor,15000.0


In [21]:
# Return the 3rd highest DISTINCT price among products as a single value.
# Note the tie: Desk and Chair are both 8000.
q("""
SELECT DISTINCT price FROM (Select price, RANK() OVER(order by price DESC) as 'rnk' from products)
as ranked where ranked.rnk=3;
""")

,price
0,8000.0


In [25]:
# For each category, return the name of the most expensive product and its price.
# If a category ties for most expensive, return all tied products.
# Products with no category are excluded.
q("""
SELECT c.category_name,p.product_name, MAX(p.price) as 'max_price' from products p
join categories c on c.category_id = p.category_id
group by c.category_name;
""")

,category_name,product_name,max_price
0,Electronics,Laptop,60000.0
1,Furniture,Desk,8000.0
2,Stationery,Notebook,50.0


In [28]:
# List each rep's name and their manager's name.
# Reps with no manager (Arjun) must still appear, manager shown as NULL.
# Sort by rep name.
q("""
SELECT e.rep_name,m.rep_name as 'manager_name' from reps e
LEFT JOIN reps m on e.manager_id = m.rep_id
ORDER BY e.rep_name ASC;
""")

,rep_name,manager_name
0,Arjun,None
1,Bhavna,Arjun
2,Charan,Arjun
3,Divya,Charan
4,Eshan,Arjun


In [32]:
# Find reps who have never made a sale (any status).
# Return rep name and region.
q("""
SELECT rep_name,region from reps where rep_id NOT IN (select rep_id from sales where rep_id is NOT NULL);
""")

,rep_name,region
0,Eshan,West


In [34]:
# For rep_id 2 (Bhavna), list each sale date, the sale revenue (quantity * price),
# and a running total of revenue over time, counting ONLY Completed sales.
# Order by date.
q("""
SELECT s.sale_date,SUM(s.quantity * p.price) over(order by s.sale_date ASC) as 'running_total' from sales s
JOIN products p on p.product_id = s.product_id
where s.rep_id = '2' and s.status = 'Completed'
""")

,sale_date,running_total
0,2024-01-05,60000.0
1,2024-01-20,70000.0
2,2024-03-03,130000.0
3,2024-04-18,146000.0


In [36]:
# Count completed sales and total completed revenue per month.
# Return month as 'YYYY-MM', the sale count, and total revenue. Sort by month.
# Hint: SQLite uses strftime('%Y-%m', sale_date), not DATE_FORMAT.
q("""
SELECT 
    strftime('%Y-%m', s.sale_date) AS month,
    COUNT(s.sale_id) AS sale_count,
    SUM(s.quantity * p.price) AS total_revenue
FROM sales s
join products p on p.product_id = s.product_id
WHERE status = 'Completed'
GROUP BY strftime('%Y-%m', sale_date)
ORDER BY month ASC;
""")

,month,sale_count,total_revenue
0,2024-01,2,70000.0
1,2024-02,2,77000.0
2,2024-03,3,90500.0
3,2024-04,1,16000.0
4,2024-05,2,84000.0
5,2024-06,2,16400.0
